# M32895 Big Data Applications - CW2
# Garbage Image Classification Using a CNN
---
**Student ID**: *UP2110919*

**Dataset**: [Garbage Classification - Kaggle](https://www.kaggle.com/datasets/asdasdasasdas/garbage-classification)

**Main Objective**: Build, train and evaluate a Convolutional Neural Network (CNN) to classify images of waste into 6 categories, using a full ML pipeline from data collection through to individual prediction.

---

## Notebook Structure

| Step | Section |
|------|---------|
| 1 | Data Collection, Validation & Preparation |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | Baseline CNN Model |
| 4 | Model Evaluation (Baseline) |
| 5 | Optimised CNN Model |
| 6 | Model Evaluation (Optimised) |
| 7 | Model Comparison & Selection |
| 8 | Prediction on Unseen Data |

---
## Step 1 - Data Collection, Validation & Preparation

### 1.1 - Import Libraries

pip install kagglehub
pip install numpy
pip install pandas matplotlib seaborn plotly tensorflow scikit-learn

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import tensorflow as tf
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Conv2D, MaxPool2D, Flatten, Dense, Dropout, BatchNormalization)
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
from PIL import Image

# suppresing certain areas of the tensor flow logs
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')

print("All libraries imported successfully.")
print(f"TensorFlow version: {tf.__version__}")

### 1.2 - Dataset Overview

The **Garbage Classification** dataset was downloaded from Kaggle. It contains **6 waste categories**:

| Label | Class |
|-------|-------|
| 0 | cardboard |
| 1 | glass |
| 2 | metal |
| 3 | paper |
| 4 | plastic |
| 5 | trash |

Images are stored in sub-folders per class under `Garbage classification/Garbage classification/`. We load them using `ImageDataGenerator`.

In [ ]:
# setting the data path and ensuring that all the classess (sub-folders) are identified nad then lists them

DATASET_PATH = "Garbage classification/Garbage classification"

# Confirm folder exists and list classes
if os.path.exists(DATASET_PATH):
    class_names = sorted(os.listdir(DATASET_PATH))
    class_names = [c for c in class_names if os.path.isdir(os.path.join(DATASET_PATH, c))]
    print(f'Dataset found: {DATASET_PATH}')
    print(f'Number of classes: {len(class_names)}')
    print(f'Classes: {class_names}')
else:
    print(f'ERROR: Path not found — please update DATASET_PATH')

### 1.3 - Data Validation

Before loading images into the model pipeline, we need validate the dataset to check for:
- The correct file formats (.jpg, .jpeg, .png only)
- Corrupt or unreadable images
- Empty class folders

In [ ]:
VALID_EXTENSIONS = ('.jpg', '.jpeg', '.png') # defining the three correct extensions

def validate_dataset(root_path, class_names):
    """
    Validates each image file in the dataset.
    Checks for:
    * Valid file extension (.jpg, .jpeg, .png)
    * File can be opened without errors (not corrupt)
    Reports a summary table of valid/invalid counts per class.
    """
    summary = []
    total_valid = 0
    total_invalid = 0

    for class_name in class_names:
        class_path = os.path.join(root_path, class_name)
        valid_count = 0
        invalid_count = 0

        for filename in os.listdir(class_path):
            filepath = os.path.join(class_path, filename)

            # Check file extension
            if not filename.lower().endswith(VALID_EXTENSIONS):
                print(f'  [INVALID EXT] {class_name}/{filename}')
                invalid_count += 1
                continue

            # Try opening the image to check it is not corrupt
            try:
                img = Image.open(filepath)
                img.verify()  # Checks integrity without fully loading
                valid_count += 1
            except Exception as e:
                print(f'  [CORRUPT] {class_name}/{filename}: {e}')
                invalid_count += 1

        total_valid += valid_count
        total_invalid += invalid_count
        summary.append({'Class': class_name, 'Valid': valid_count, 'Invalid': invalid_count})

    df_summary = pd.DataFrame(summary)
    print('\n--- Validation Summary ---')
    print(df_summary.to_string(index=False))
    print(f'\nTotal valid   : {total_valid}')
    print(f'Total invalid : {total_invalid}')
    return df_summary

df_validation = validate_dataset(DATASET_PATH, class_names)

### 1.4 - Load Images into NumPy Arrays

We load every image from thier sub-folders, resize them to 100×100 pixels, and store them as a NumPy array

We also record the integer class label for each image.

In [ ]:
IMG_SIZE = (100, 100) # Specifying the X and Y pixel count

def load_images(root_path, class_names, img_size):
    """
    Loads all images from class sub-folders into NumPy arrays.
    Resizes every image to img_size (width, height).
    Returns:
    * X — NumPy array of shape (n_images, height, width, 3)
    * y — NumPy array of integer labels
    """
    X = []
    y = []

    for label_idx, class_name in enumerate(class_names):
        class_path = os.path.join(root_path, class_name)
        loaded = 0

        for filename in os.listdir(class_path):
            if not filename.lower().endswith(VALID_EXTENSIONS):
                continue
            filepath = os.path.join(class_path, filename)
            try:
                img = Image.open(filepath).convert('RGB')  # Ensure the photo has 3 colour channels
                img = img.resize(img_size)                 # Resize it to the target size
                arr = np.array(img)                        # Convert it to a NumPy array
                X.append(arr)
                y.append(label_idx)
                loaded += 1
            except Exception:
                pass  # Skip past any corrupt images

        print(f'  Loaded {loaded} images for class: {class_name}')

    X = np.array(X)  # Shape: (n_images, 100, 100, 3)
    y = np.array(y)  # Shape: (n_images,)
    return X, y

print('Loading images...')
X, y = load_images(DATASET_PATH, class_names, IMG_SIZE) # run the defined function "load_images" supplying the correct inputs

print(f'\nX shape: {X.shape}') # print the number of images, the size of each and their colour "depth".
print(f'y shape: {y.shape}') # print the number of arrays
print(f'Data type: {X.dtype}') # print the data type

### 1.5 - Check a Sample Image

using a `pointer` to inspect one image from the array and confirm it looks correct before moving on

In [ ]:
pointer = 10  # the index of a random image

print(f'Array pointer  = {pointer}')
print(f'X[{pointer}] shape : {X[pointer].shape}')  # Should be (100, 100, 3)
print(f'Label          : {y[pointer]} ({class_names[y[pointer]]})')

plt.imshow(X[pointer])
plt.title(f'Sample image — class: {class_names[y[pointer]]}')
plt.axis('off')
plt.show()

### 1.6 - Validate the Loaded Arrays

We validate the loaded NumPy arrays to confirm:
- Every element is a NumPy array
- Every image has the correct shape
- Pixel values are in the expected range (0–255)
- No Not a Number (NaN) values are present

In [ ]:
def check_images(dataset, dataset_name):
    """
    Checks images for:
    * Being a NumPy array
    * Correct shape
    * Pixel values in range 0-255
    * No NaN values
    """
    invalid_count = 0
    valid_count = 0

    for idx, image in enumerate(dataset):
        # Check it is a NumPy array
        if not isinstance(image, np.ndarray):
            print(f'{dataset_name} - Index {idx}: Not a valid array')
            invalid_count += 1
            continue

        # Check shape is (100, 100, 3)
        if image.shape != (100, 100, 3):
            print(f'{dataset_name} - Index {idx}: Incorrect shape {image.shape}')
            invalid_count += 1
            continue

        # Check pixel values are in valid range
        if not (image.min() >= 0 and image.max() <= 255):
            print(f'{dataset_name} - Index {idx}: Invalid pixel values')
            invalid_count += 1
            continue

        # Check for NaN values
        if np.isnan(image.astype(float)).any():
            print(f'{dataset_name} - Index {idx}: Contains NaN values')
            invalid_count += 1
            continue

        valid_count += 1

    print(f'\n{dataset_name}: {valid_count} valid images, {invalid_count} invalid images')

print('Checking images...\n')
check_images(X, 'Full dataset') # running the defined function and printing valid/invlaid images

### 1.7 — Train / Validation / Test Split

We now need to split the data into **train**, **validation**, and **test** sets.

- **Train (80%)** — used to train the model
- **Validation (10%)** — used during training to monitor performance and trigger early stopping
- **Test (10%)** — kept completely unseen until final evaluation

In [ ]:
# First split: 80% train, 20% temp
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.20, random_state=0)

# Second split: temp 50/50 into validation and test
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.50, random_state=0)

print(f'* Train set      : {X_train.shape}, {y_train.shape}')
print(f'* Validation set : {X_val.shape}, {y_val.shape}')
print(f'* Test set       : {X_test.shape}, {y_test.shape}')

### 1.8 - Rescale and Reshape 
Now we must:
1. Rescale pixel values from [0, 255] tp [0, 1] by dividing by 255. this normalises inputs and helps the model train faster and more stable.

| Before Scaling | After Scaling |
|----------------|---------------|
| Pixel values: 0–255 | Pixel values: 0.0–1.0 |
| Integer type (uint8) | Floating-point (float32) |
| Can cause instability in training | Helps stable and faster training |

In [ ]:
# Check current max pixel value before scaling
print(f'Max pixel value before scaling: {X_train.max()}')

# Convert to float32 and rescale to [0, 1]
X_train = X_train.astype('float32') / 255.0
X_val   = X_val.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0

print(f'Max pixel value after scaling: {X_train.max()}')

In [ ]:
# Convert integer labels to an encoded categorical format (e.g. 0, 0, 0, 0, 1, 0)
N_LABELS = len(class_names)  # for all 6 classes

y_train = to_categorical(y_train, num_classes=N_LABELS)
y_val   = to_categorical(y_val,   num_classes=N_LABELS)
y_test  = to_categorical(y_test,  num_classes=N_LABELS)

print(f'y_train shape after encoding: {y_train.shape}')
print(f'Example encoded label: {y_train[0]}')

---
## Step 2 - Exploratory Data Analysis (EDA)

### 2.1 - Class Distribution

We need to check if the dataset is balanced. An imbalanced dataset can bias the model toward the majority class. This causes poor performance on rarer classes.

In [ ]:
# after "one-hot" encoding the y values, we need to use argmax to get the labels back to integers for counting.
y_train_int = np.argmax(y_train, axis=1)
y_val_int   = np.argmax(y_val,   axis=1)
y_test_int  = np.argmax(y_test,  axis=1)

# Creating a dataframe for label frequency distribution
df_freq = pd.DataFrame(columns=['Set', 'Label', 'Frequency'])

def count_labels(y_int, dataset_name):
    """
    Counts occurrences of each label and adds to df_freq.
    """
    global df_freq
    unique, counts = np.unique(y_int, return_counts=True)
    for label, frequency in zip(unique, counts):
        df_freq = pd.concat([
            df_freq,
            pd.DataFrame([{'Set': dataset_name, 'Label': class_names[label], 'Frequency': frequency}])
        ], ignore_index=True)
        print(f'* {dataset_name} — {class_names[label]}: {frequency} images')

count_labels(y_train_int, 'Train')
count_labels(y_val_int,   'Validation')
count_labels(y_test_int,  'Test')

In [ ]:
# now we can visualise the frequency ditribution of the various labels
plt.figure(figsize=(14, 6))
sns.barplot(data=df_freq, x='Set', y='Frequency', hue='Label')
plt.xticks(rotation=45)
plt.title('Label Frequency Distribution in Train, Validation and Test Sets')
plt.xlabel('Dataset Split')
plt.ylabel('Number of Images')
plt.tight_layout()
plt.show()

This graph shows the frequency distribution of the various different labels and gives valuable information of the weighting of the classes. The Train, Validation, and Test sets all showcase similar proportions of the various classes.

### 2.2 - Sample Images Per Class

By displaying one sample image from each class we can identify ones that may be confused by the model. 

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(16, 4))
axes = axes.flatten()

for idx in range(N_LABELS):
    # Find first image in X_train that belongs to this class
    class_indices = np.where(y_train_int == idx)[0]
    sample_idx = class_indices[0]

    axes[idx].imshow(X_train[sample_idx])
    axes[idx].set_title(class_names[idx], fontsize=11)
    axes[idx].axis('off')

plt.suptitle('Sample Image from Each Waste Class', fontsize=14)
plt.tight_layout()
plt.show()

### 2.3 - Average image per class

By computing and displaying the average (mean) pixel values accross all images in each class we can teach the model what colour and rough shape to begin associating with each category.

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(16, 4))
axes = axes.flatten()

for idx in range(N_LABELS):
    class_images = X_train[y_train_int == idx]  # All images for this class
    avg_image = np.mean(class_images, axis=0)   # Mean across all images

    axes[idx].imshow(avg_image)
    axes[idx].set_title(f'{class_names[idx]} (n={len(class_images)})', fontsize=10)
    axes[idx].axis('off')

plt.suptitle('Average Image Per Class (Train Set)', fontsize=14)
plt.tight_layout()
plt.show()

This set of figures does not show us an awful lot but a clear variation in the average pixel count can be observed.

---
## Step 3 - Baseline CNN Model

### Why use a CNN?

A Convolutional Neural Network (CNN) is the standard model for image classification. Unlike a Dense, fully connected, network a CNN uses convolutional layers to learn spacial features (edges, textures, and shapes) directly from pixel data. This is much more effective than traditional machine learning models (decision trees or logistics regression) which cannot process raw pixel grids in a meaninful way.

### Baseline Architecture

We start with a simple two-block baseline to establish a performance floor before optimisation.

| Layer | Purpose |
|-------|---------|
| Conv2D (32 filters) | Learns low-level features: edges and corners |
| MaxPool2D | Reduces spatial size, keeps dominant features |
| Conv2D (64 filters) | Learns higher-level features: textures and patterns |
| MaxPool2D | Further spatial reduction |
| Flatten | Converts 2D feature maps into a 1D vector |
| Dense (128 units) | Combines features to learn class patterns |
| Dropout (0.5) | Randomly disables 50% of neurons per step — reduces overfitting |
| Dense (12, softmax) | Outputs a probability for each of the 12 classes |

In [ ]:
def build_tf_model(input_shape, n_labels):
    model = Sequential()

    model.add(Conv2D(filters=32, kernel_size=(3,3), input_shape=input_shape, activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    model.add(Conv2D(filters=64, kernel_size=(3,3), activation='relu'))
    model.add(MaxPool2D(pool_size=(2, 2)))

    model.add(Flatten())

    model.add(Dense(128, activation='relu'))
    model.add(Dropout(0.5))

    model.add(Dense(n_labels, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    return model

baseline_model = build_tf_model(input_shape=X_train.shape[1:], n_labels=N_LABELS)
baseline_model.summary()

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', mode='min', verbose=1, patience=5)

baseline_model = build_tf_model(input_shape=X_train.shape[1:], n_labels=N_LABELS)

baseline_model.fit(
    x=X_train,
    y=y_train,
    epochs=10,
    validation_data=(X_val, y_val),
    verbose=1,
    callbacks=[early_stop]
)

---
## Step 4 - Model Evaluation (Baseline)

### Training curves

We plot loss and accuracy over the number of epochs for both the training and validation sets.

- Loss will show how wrong the model is (the lower the better). A healthy model both of the curves falling together. Overfitting occurs when the validation loss starts rising when training loss keeps falling.
- The Accuracy measures the proportion of correctly classified images (higher the better). Training and vlaidation accuracy should track eahc other closely here. A large gap between validation and training curves is the key signal of overfitting.

In [ ]:
history = pd.DataFrame(baseline_model.history.history)
history.head()

In [ ]:
sns.set_style('whitegrid')

history[['loss', 'val_loss']].plot(style='.-')
plt.title('Baseline CNN — Loss')
plt.xlabel('Epoch')
plt.show()

print('\n')
history[['accuracy', 'val_accuracy']].plot(style='.-')
plt.title('Baseline CNN — Accuracy')
plt.xlabel('Epoch')
plt.show()

After running this with 10 Epochs we can see the lines diverging greatly from between 3-5 epochs

### Classification report and Confusion matrix

| Metric | Definition | Why higher is better |
|--------|------------|----------------------|
| **Accuracy** | Overall % of correctly classified images | Directly measures correct predictions |
| **Precision** | Of all images predicted as class X, how many actually were? | Fewer false positives |
| **Recall** | Of all real class X images, how many did the model find? | Fewer missed items |
| **F1-Score** | Harmonic mean of precision and recall | Balances both — useful for all classes equally |

the confusion matrix shows which classess are mostl commonly confused with each other.

In [ ]:
def confusion_matrix_and_report(X, y, pipeline, label_map):
    """
    Prints confusion matrix and classification report, and plots heatmap.
    """
    # Predictions (convert from one-hot)
    prediction = pipeline.predict(X)
    prediction = np.argmax(prediction, axis=1)

    # True labels (convert from one-hot)
    y = np.argmax(y, axis=1)

    # Compute confusion matrix
    cm = confusion_matrix(y_true=y, y_pred=prediction)

    print('---  Confusion Matrix  ---')
    print(pd.DataFrame(
        cm,
        columns=['Actual ' + sub for sub in label_map],
        index=['Predicted ' + sub for sub in label_map]
    ))
    print('\n')

    print('---  Classification Report  ---')
    print(classification_report(y, prediction, target_names=label_map), '\n')

    # Plot heatmap
    plt.figure(figsize=(8, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=label_map,
        yticklabels=label_map
    )
    plt.xlabel('Actual')
    plt.ylabel('Predicted')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.show()

In [ ]:
# defining the function to plot all of the confusion matrix's for each the train, test, and validation sets.
def clf_performance(X_train, y_train, X_val, y_val, X_test, y_test, pipeline, label_map):
    """
    Runs confusion_matrix_and_report on all three sets.
    """
    print('#### Train Set ####\n')
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

    print('#### Validation Set ####\n')
    confusion_matrix_and_report(X_val, y_val, pipeline, label_map)

    print('#### Test Set ####\n')
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map)

In [ ]:
clf_performance(X_train, y_train,
                X_test,y_test,
                X_val, y_val,
                baseline_model,
                label_map= class_names
                )

---
## Step 5 - Optimised CNN Model

### Optimisation strategy

Based on the baseline results, the following targeted improvements will be made:

---
## Step 6 - Model Evaluation (Optimised)

---
## Step 7 - Model Comparison and Selection

---
## Step 8 - Prediction on Unseen Data